# Multi-head Latent Attention (MLA)

Multi-head Latent Attention projects hidden states into a low-rank latent
vector instead of separate K and V matrices. Only the latent is cached, which
cuts KV cache by 93% on DeepSeek-V2 numbers.

This is the full **PyTorch** version of the [LLM Quest](https://bankoti.github.io/llm-quest)
browser challenge. Fill in each TODO, then run the checks cell.

Runs on the free Colab CPU runtime; no GPU needed for this exercise.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

B, T, C = 2, 5, 16   # batch, seq_len, hidden_dim
d_c = 4              # latent dimension (compressed KV)
H_kv, D = 2, 3       # KV heads and head dimension

In [ ]:
# TODO 1: compress hidden states to KV latent
# x: (B, T, C) -> c_KV: (B, T, d_c)
W_DKV = nn.Linear(C, d_c, bias=False)

def compress_kv(x, W_DKV):
    # one matmul: x @ W_DKV.weight.T
    raise NotImplementedError

In [ ]:
# TODO 2: reconstruct K and V from the latent
# c_kv: (B, T, d_c) -> K: (B, T, H_kv*D), V: (B, T, H_kv*D)
W_UK = nn.Linear(d_c, H_kv * D, bias=False)
W_UV = nn.Linear(d_c, H_kv * D, bias=False)

def expand_kv(c_kv, W_UK, W_UV):
    raise NotImplementedError

In [ ]:
# TODO 3: MLA cache bytes
# Only c_KV is cached, shape (batch, seq_len, d_c) per layer.
def mla_cache_bytes(seq_len, d_c, n_layers, batch_size, bytes_per_element=2):
    raise NotImplementedError

# TODO 4: reduction factor vs standard GQA
# GQA caches K and V: 2 * n_kv_heads * head_dim bytes per token
# MLA caches c_KV: d_c bytes per token
def cache_reduction_factor(n_kv_heads, head_dim, d_c):
    raise NotImplementedError

In [ ]:
# Checks
x_t = torch.randn(B, T, C)
c_kv = compress_kv(x_t, W_DKV)
assert c_kv.shape == (B, T, d_c), f"latent shape: {c_kv.shape}"

K, V = expand_kv(c_kv, W_UK, W_UV)
assert K.shape == (B, T, H_kv * D), f"K shape: {K.shape}"
assert V.shape == (B, T, H_kv * D), f"V shape: {V.shape}"

# DeepSeek-V2 numbers: d_c=512, 128 KV heads, D=128 -> 64x reduction
ratio = cache_reduction_factor(128, 128, 512)
assert abs(ratio - 64.0) < 1e-4, f"reduction ratio: {ratio}"

cache = mla_cache_bytes(seq_len=1024, d_c=512, n_layers=4, batch_size=2)
expected = 4 * 2 * 1024 * 512 * 2
assert cache == expected, f"cache bytes: {cache} != {expected}"

print("All MLA checks passed.")
print(f"DeepSeek-V2 style: {ratio:.0f}x cache reduction vs standard KV.")